In [94]:
import re
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer

### 복습
- 가전 폴더 안에 모든 데이터파일을 로드해서 하나의 데이터프레임으로 생성
- 감정에 대한 데이터들이 3개 분류 -> 2개 분류로 변경 (부정, 중립 -> 부정)
- 감정 데이터가 없는 데이터들은 따로 저장
- train, test의 비율은 8:2
- Dataset을 구성할때 생성자 함수에서는 데이터는 그냥 self변수에 저장
- '__getitem__' 함수에서 임베딩 후 되돌려주는 형태로 구성 변경
- RawText 데이터를 이용하여 감정분석 모델을 생성
- SBERT 모델을 이용하여 임베딩
- 다중퍼셉트론의 모델을 이용하여 감정 분석 (Linear -> ReLU -> Dropout -> Linear)
- 검증 데이터를 이용하여 정확도와 f1_score 확인
- 감정 데이터가 없는 RawText에서 sample(10)를 출력하여 감정 예측
- 다중퍼셉트론 모델이 아닌 머신러닝 모델 (SVC)을 이용하여 감정 분석 예측

In [95]:
# 모든 가전 폴더의 데이터들의 단순 행 결합을 통한 데이터프레임 생성
df = pd.DataFrame()

# 폴더 안에 파일 불러와서 데이터프레임에 추가
for i in range(1, 5):
    if i == 1:
        for j in range(76, 89):
            data = pd.read_json(f"../data/가전/3-{i}.영상음향가전({j}).json")
            df = pd.concat([df, data], ignore_index=True)
    elif i == 2:
        for j in range(128, 141):
            data = pd.read_json(f"../data/가전/3-{i}.생활미용욕실가전({j}).json")
            df = pd.concat([df, data], ignore_index=True)
    elif i == 3:
        for j in range(127, 140):
            data = pd.read_json(f"../data/가전/3-{i}.주방가전({j}).json")
            df = pd.concat([df, data], ignore_index=True)
    else:
        for j in range(126, 129):
            data = pd.read_json(f"../data/가전/3-{i}.계절가전({j}).json")
            df = pd.concat([df, data], ignore_index=True)
            

df.head(1)

,Index,RawText,Source,Domain,MainCategory,ProductName,ReviewScore,Syllable,Word,RDate,GeneralPolarity,Aspects
0,112038,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,SNS,가전,영상/음향가전,(1+1세트) TJ 태진 블루투스 마이크 / 무선 노래방 마이크,3,323,73,20221110,0.0,"[{'Aspect': '제조일/제조사', 'SentimentText': '우리나라 ..."


In [96]:
df_na = df[df['GeneralPolarity'].isna()]
len(df_na)

378

In [97]:
df2 = df.dropna(subset=['GeneralPolarity']).reset_index(drop=True)
len(df2)

3678

In [98]:
df3 = df2[['RawText', 'GeneralPolarity']]
df3.rename(columns={'GeneralPolarity': 'labels'}, inplace=True)
df3

C:\Users\abohv\AppData\Local\Temp\ipykernel_11044\816840456.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3.rename(columns={'GeneralPolarity': 'labels'}, inplace=True)


,RawText,labels
0,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,0.0
1,누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요ㅠㅠ일단 겉 부분에...,-1.0
2,노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...,0.0
3,너무 별로예요 ㅡㅡ… 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요… 음질이 거...,-1.0
4,소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...,-1.0
...,...,...
3673,사진으로는 잘 체감하지 못했는데 실제 실물을 보니 디자인이 조금 충격적이라고나 할까...,1.0
3674,이 에어컨의 제일 큰 장점은 조작법에 있는 것 같아요.인공지능 조작이 가능해서 한번...,1.0
3675,"이 제품을 추천하는 이유는요! 일단 LED를 통해 공기질을 눈으로 확인할 수 있고,...",1.0
3676,"인터넷에서 구매하였는데요, 이 공기청정기 처음 발견하고, 처음에는 오잉? 이게뭐지?...",1.0


In [99]:
def noramlize(text):
    text= re.sub(r'[^가-힣a-zA-Z0-9\s\.]', " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df3['RawText'] = df3['RawText'].map(noramlize)

C:\Users\abohv\AppData\Local\Temp\ipykernel_11044\3682444055.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['RawText'] = df3['RawText'].map(noramlize)


In [100]:
# 결측치를 제거
df3.dropna(subset='RawText', inplace=True)

C:\Users\abohv\AppData\Local\Temp\ipykernel_11044\915550961.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3.dropna(subset='RawText', inplace=True)


In [101]:
flag = df3['RawText'].str.len() > 1
df3.loc[flag, ]

,RawText,labels
0,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,0.0
1,누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요 일단 겉 부분에 ...,-1.0
2,노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...,0.0
3,너무 별로예요 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요 음질이 거의 입 안...,-1.0
4,소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...,-1.0
...,...,...
3673,사진으로는 잘 체감하지 못했는데 실제 실물을 보니 디자인이 조금 충격적이라고나 할까...,1.0
3674,이 에어컨의 제일 큰 장점은 조작법에 있는 것 같아요.인공지능 조작이 가능해서 한번...,1.0
3675,이 제품을 추천하는 이유는요 일단 LED를 통해 공기질을 눈으로 확인할 수 있고 터...,1.0
3676,인터넷에서 구매하였는데요 이 공기청정기 처음 발견하고 처음에는 오잉 이게뭐지 했어요...,1.0


In [102]:
# for i in range(len(df3)):
#     if df3.loc[i, 'labels'] == -1 or df3.loc[i, 'labels'] == 0:
#         df3.loc[i, 'labels'] = 0
#     else:
#         df3.loc[i, 'labels'] = 1
df3['labels'] = df3['labels'].astype(int) + 1

C:\Users\abohv\AppData\Local\Temp\ipykernel_11044\167276577.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['labels'] = df3['labels'].astype(int) + 1


In [103]:
train_df, test_df = train_test_split(df3, test_size=0.2, random_state=42, stratify=df3['labels'])

In [104]:
model_name3 = 'BM-K/KoSimCSE-roberta-multitask'

sbert3 = SentenceTransformer(model_name3) 

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [105]:
class SBERTDataset(Dataset):
    def __init__(self, document, labels, max_length=128):
        self.max_length = max_length
        sbert3.max_seq_length = self.max_length
        with torch.inference_mode():
            self.emb = sbert3.encode(
                document,
                convert_to_tensor=True,
                normalize_embeddings=True
            )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.emb[idx], self.labels[idx]
    
# Dataset의 형태로 데이터프레임을 변환
train_ds = SBERTDataset(train_df['RawText'].tolist(), train_df['labels'].tolist(), max_length=128)
test_ds = SBERTDataset(test_df['RawText'].tolist(), test_df['labels'].tolist(), max_length=128)

In [106]:
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=128, shuffle=True)

In [ ]:
class MLPHead(nn.Module):
    # 선형 모델에 데이터를 입력하는 형태
    # 선형 모델을 정의할때 인자값 (Linear(입력데이터의 피쳐의 수, 출력의 피쳐의 수))
    def __init__(self, input_dim, hidden=256, num_classes=3):
        super().__init__()
        # 다중 퍼셉트론층 구성
        self.net = nn.Sequential(
            # 선형 모델
            nn.Linear(input_dim, hidden),   # 입력 768 차원에서 출력은 256 차원
            nn.ReLU(),                      # 비선형 구조를 이해하기 위한 작업
            nn.Dropout(0.2),               # 과적합 방지를 위한 소실 작업
            nn.Linear(hidden, num_classes)  # 최종 출력층
            
        )
    # 순전파 함수 -> 독립변수를 받아서 예측 값을 되돌려준다.
    def forward(self, x):
        # x : 독립 변수 (document 데이터를 임베딩하고 배치로 묶은 데이터)
        result = self.net(x)    # 출력이 2차원인 확률 데이터
        return result


In [108]:
in_dim = sbert3.get_sentence_embedding_dimension()  # 출력 피쳐의 수를 되돌려주는 내장함수
in_dim

768

In [109]:
# MLPHead 모델 생성
clf = MLPHead(in_dim)
# 손실함수 -> 예측값과 실제값의 차이를 확인하는 함수
crit = nn.CrossEntropyLoss()
# 옵티마이저
opt = torch.optim.AdamW(clf.parameters(), lr= 2e-4)

In [110]:
# 학습 루프
# 학습 모드 전환
clf.train()

for epoch in range(5):
    total = 0.0
    for x, y in train_dl:
        # x : document데이터가 임베딩 벡터가 된 묶음 (tensor)
        # y : labels데이터가 tensor혀앹 묶음
        opt.zero_grad()
        # 순전파
        logits = clf(x)
        # 손실 계산 (예측값, 실제값)
        loss = crit(logits, y)
        # 역전파
        loss.backward()
        
        # 스탭
        opt.step()
        total += loss.item() * float(x.size(0))
    print(f"epoch : {epoch}, loss : {total/len(train_ds)}")

epoch : 0, loss : 1.0566214719327593
epoch : 1, loss : 0.9580445629573047
epoch : 2, loss : 0.8892263642466044
epoch : 3, loss : 0.8435388588727047
epoch : 4, loss : 0.8001496583972959


In [112]:
clf.eval()


y_true, y_pred = [], []


with torch.inference_mode():
    for x, y in test_dl:
        logits = clf(x)     # 예측 데이터 -> [0.xxx, 0.xxx]
        pred = logits.argmax(dim=1).tolist()     # 예측 데이터 -> [0, 1, 1, 0, ...]
        # y_true에 y를 리스트의 형태로 변환하고 데이터를 학장시킨다
        y_true.extend(y.tolist())
        y_pred += pred

print("f1_score", f1_score(y_true, y_pred, average='macro'))

f1_score 0.3290267011197244


In [115]:
samples = df_na.head(10)['RawText'].tolist()


id2label = {
    0 : '부정',
    1 : '중립',
    2 : '긍정'
}

@torch.no_grad()
def predict_review(
    texts, 
    batch_size = 128
):
    # texts : 예측하려고 하는 리뷰의 원문 데이터들
    # batch_size : 묶음의 크기 

    # texts의 정규화 -> texts(list형태) -> map(), for문을 이용하여 정규화 
                #   -> texts(str) -> 1. 문자열을 정규화 함수에 입력, 
                # 2. 문자열이면 리스트의 형태로 변환
    if isinstance(texts, str):
        texts = [texts]
    
    # 정규화 함수에 리뷰 데이터를 넣어준다. 
    texts_norm = [noramlize(t) for t in texts]

    # 2개의 모델을 평가모드 전환 clf, sbert3
    sbert3.eval()
    clf.eval()

    # 결과 값
    result = []

    # 배치 데이터로 구성 -> encode -> 분류 모델에 데이터 입력 -> 출력 값을 설정 -> result에 대입
    for idx in range(0, len(texts_norm), batch_size):
        batch_texts = texts_norm[ idx : idx + batch_size ]
        # texts = ['a', b', 'c'] 
        # batch_size = 2
        # 첫번째 반복 구간에서는 
        # batch_texts -> ['a', 'b']
        # embs -> 벡터화 -> 열의 개수는 sbert3의 아웃풋의 차원의 수(768) 
        #               -> 행의 개수는 len(batch_texts)
        # probs -> [ [ 0.3, 0.7 ] , [ 0.51, 0.49 ]]
        # preds -> [ 1 , 0 ]
        # sbert3의 encode함수를 이용하여 임베딩 벡터 생성 
        embs = sbert3.encode(
            batch_texts, 
            convert_to_tensor=True, 
            normalize_embeddings=True
        )
        # embs를 clf 모델을 이용하여 예측 확률 데이터를 생성 
        logits = clf(embs)
        probs = logits.softmax(dim = -1)
        preds = probs.argmax(dim=-1).tolist()

        for idx2, pred in enumerate(preds):
            # 첫번째 반복문의 1번 루프에서 preds -> [1, 0]
            # 두번째 반복문의 첫번째 루프 
            # idx2 -> 0
            # pred -> 1
            # prob -> prods[0, 1] -> 0.7
            # review -> texts[ 0 + 0 ] -> texts[0]
            # idx2 : 인덱스 
            # pred : 예측 값(예측 확률의 인덱스 - 확률이 높은 곳의 인덱스(0,1))
            # 높은 예측율
            prob = float( probs[idx2, pred] )
            # 리뷰의 원문 
            # 첫번째 반복문의 반복 횟수? -> len(texts) / batch_size + 1
            # 두번째 반복문의 반복 횟수? -> 
            # idx -> 배치의 시작지점
            # idx2 -> 시작점부터 얼마만큼 이동했는가?
            review = texts[idx + idx2]
            # 긍정/부정 라벨링
            label = id2label[pred]
            # prob, review, label들을 result에 추가 
            result.append(
                {
                    'text' : review, 
                    'prob' : prob, 
                    'label' : label
                }
            )
    return result

In [116]:
out_data = predict_review(samples)
out_data

[{'text': '귀에서 자꾸 빠져요.귀에 꼽는재 질이 미끄러운 재질이라 작은 소품이지만 재질만 바꾸어도 안 빠질것 같은데 000 것은 귀에 꼽으면 안 빠져서 사용할 때 아무 문제 없고 멍멍한 현상도 없는데 아무튼 만드는 이가 000 것과 사용 비교 해보고 하다 못해 재질이라도 바꾸든지 반품도 못하고 통화 중 몇 번씩 빠져서 사용 불가능 합니다.통화음은 상대방쪽 발음이 잘 안들린다 하고 음악 들을때는 스테레오 됩니다.통화시 사용하는 사람에게는 음질이 안 좋아 비추천내쪽에서 말하는것이 상대가 잘 발음이 안들린다 해서 중간에 통화를 중단하는 경우가 여러번 대기업 회사에서 시착도 안해보고 완성된 물건이라 판매 하는지그냥 만드는건지 ...오픈해서 반품도 못하고 너무하네요.',
  'prob': 0.398764044046402,
  'label': '부정'},
 {'text': '아이를 출산한 기념으로 TV를 바꿨습니다. 그전에 쓰던 TV가 꽤나 무거워서 떨어지거나 하면 아이가 다칠까 봐 걱정이 되었거든요. 그런데 이 TV는 마감 퀄리티가 별로입니다. 아이가 자칫하다 TV를 만지면 손이 베일까 봐 걱정이에요. 저희 남편은 연결을 하다가 손이 살짝 까졌습니다.전에 쓰던 TV보다 무게가 가벼워서, 아이가 혹여나 다칠 일이 줄어들지 않을까 구입한 TV인데 마감이 별로라 더 걱정이네요. 마감이 깔끔하지 못한 부분은 사포로 살짝 갈아볼까 하는데 그렇게 되면 디자인도 떨어지게 될까 봐 노심초사하는 중입니다. 처음부터 마감이 괜찮았으면 이런 걱정도 없었을 텐데 여러모로 아쉽네요.',
  'prob': 0.43911561369895935,
  'label': '긍정'},
 {'text': '화면에 노이즈가 생깁니다. 저희 가족 중에 아무도 TV 화면을 건드리거나 하지 않았는데 사용한 지 한 달이 되는 지금, 갑자기 화면에 노이즈가 생기네요. 큰맘 먹고 구입한 TV라 혹여나 화면에 흠집이라도 생길까 봐 조심히 청소했습니다. 배신감이 좀 드네요.이 TV의 가장 큰 장점이 화질이라고 생

------

## 강사님 Ver

In [167]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer

In [168]:
# 문서 정규화 함수 정의
def normalize(text):
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [169]:
# 파일의 목록을 로드 -> 목록을 기준으로 데이터를 로드 -> 단순 결합
file_path = "../data/가전/"
file_list = os.listdir(file_path)
file_list

['3-1.영상음향가전(76).json',
 '3-1.영상음향가전(77).json',
 '3-1.영상음향가전(78).json',
 '3-1.영상음향가전(79).json',
 '3-1.영상음향가전(80).json',
 '3-1.영상음향가전(81).json',
 '3-1.영상음향가전(82).json',
 '3-1.영상음향가전(83).json',
 '3-1.영상음향가전(84).json',
 '3-1.영상음향가전(85).json',
 '3-1.영상음향가전(86).json',
 '3-1.영상음향가전(87).json',
 '3-1.영상음향가전(88).json',
 '3-2.생활미용욕실가전(128).json',
 '3-2.생활미용욕실가전(129).json',
 '3-2.생활미용욕실가전(130).json',
 '3-2.생활미용욕실가전(131).json',
 '3-2.생활미용욕실가전(132).json',
 '3-2.생활미용욕실가전(133).json',
 '3-2.생활미용욕실가전(134).json',
 '3-2.생활미용욕실가전(135).json',
 '3-2.생활미용욕실가전(136).json',
 '3-2.생활미용욕실가전(137).json',
 '3-2.생활미용욕실가전(138).json',
 '3-2.생활미용욕실가전(139).json',
 '3-2.생활미용욕실가전(140).json',
 '3-3.주방가전(127).json',
 '3-3.주방가전(128).json',
 '3-3.주방가전(129).json',
 '3-3.주방가전(130).json',
 '3-3.주방가전(131).json',
 '3-3.주방가전(132).json',
 '3-3.주방가전(133).json',
 '3-3.주방가전(134).json',
 '3-3.주방가전(135).json',
 '3-3.주방가전(136).json',
 '3-3.주방가전(137).json',
 '3-3.주방가전(138).json',
 '3-3.주방가전(139).json',
 '3-4.계절가전(126).json',
 '3-4.계절가전(127)

In [201]:
# 로드한 데이터프레임을 누적으로 결합하기 위해 빈 데이터프레임 생성
df = pd.DataFrame()

for file in file_list:
    data = pd.read_json(file_path + file)
    df = pd.concat([df, data], axis=0)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 411.9+ KB


In [202]:
# 필요한 컬럼을 제외하고 나머지 컬럼을 무시
df = df[['RawText', 'GeneralPolarity']]
df

,RawText,GeneralPolarity
0,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,0.0
1,누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요ㅠㅠ일단 겉 부분에...,-1.0
2,노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...,0.0
3,너무 별로예요 ㅡㅡ… 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요… 음질이 거...,-1.0
4,소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...,-1.0
...,...,...
95,사진으로는 잘 체감하지 못했는데 실제 실물을 보니 디자인이 조금 충격적이라고나 할까...,1.0
96,이 에어컨의 제일 큰 장점은 조작법에 있는 것 같아요.인공지능 조작이 가능해서 한번...,1.0
97,"이 제품을 추천하는 이유는요! 일단 LED를 통해 공기질을 눈으로 확인할 수 있고,...",1.0
98,"인터넷에서 구매하였는데요, 이 공기청정기 처음 발견하고, 처음에는 오잉? 이게뭐지?...",1.0


In [203]:
df.rename(columns = {
    'GeneralPolarity' : 'label'
}, inplace=True)

In [204]:
df['RawText'] = df['RawText'].map(noramlize)

In [205]:
df = df.loc[df['RawText'].str.len() > 1]

In [206]:
df.drop_duplicates(subset='RawText', inplace=True)

In [207]:
na_df = df.loc[df['label'].isna(), ]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  4056 non-null   object 
 1   label    3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 95.1+ KB


In [208]:
df = df.loc[~df['label'].isna(), ]

In [209]:
df.reset_index(drop=True, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3678 entries, 0 to 3677
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  3678 non-null   object 
 1   label    3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 57.6+ KB


In [210]:
df['label'].value_counts()

label
 1.0    2220
 0.0     944
-1.0     514
Name: count, dtype: int64

In [211]:
df['label'] = df['label'].map(
    {
        -1 : 0,
        0 : 0,
        1 : 1
    }
)

In [212]:
df['label'].value_counts()

label
1    2220
0    1458
Name: count, dtype: int64

In [213]:
na_df 

,RawText,label
13,귀에서 자꾸 빠져요.귀에 꼽는재 질이 미끄러운 재질이라 작은 소품이지만 재질만 바꾸...,NaN
43,아이를 출산한 기념으로 TV를 바꿨습니다. 그전에 쓰던 TV가 꽤나 무거워서 떨어지...,NaN
44,화면에 노이즈가 생깁니다. 저희 가족 중에 아무도 TV 화면을 건드리거나 하지 않았...,NaN
55,이번에 이사하면서 우리 따님께서 방에 TV가 있으면 좋겠다고 하여 방에서 사용할 T...,NaN
93,기존에 사용하던 무선이어폰이 오래되어서 배터리가 광탈하는 바람에 새로운 상품이 필요...,NaN
...,...,...
41,대용량이고 세척이 간편하다는 얘기에 구매를 했는데 저는 별로인 거 같아요...생각했...,NaN
44,요거 진짜 진짜 물건입니다. 처음엔 디자인이 너무 귀여워서 주문하게 되었는데요. 작...,NaN
68,지인의 추천으로 믿고 바로 구매를 해서 현재도 사용중입니다좀 더 많은 분들에게 도움...,NaN
76,다른 에어쿨러와 다르게 슬림한 디자인이 마음에 들어요.심플한 디자인 덕분에 집안 어...,NaN


In [214]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

In [215]:
model_name = 'BM-K/KoSimCSE-roberta-multitask'

sbert = SentenceTransformer(model_name) 

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [216]:
max_length = 128

In [219]:
class SBERTDataset(Dataset):
    def __init__(self, document, labels, max_length=max_length):
        self.max_length = max_length
        sbert3.max_seq_length = self.max_length
        with torch.inference_mode():
            self.emb = sbert3.encode(
                document,
                convert_to_tensor=True,
                normalize_embeddings=True
            )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.emb[idx], self.labels[idx]
    # getitem에서 임베딩 처리
    # res_emb = sbert.encode(
    #     self.texts[idx], convert_to_tensor=True, normalize_embeddings=True
    #     )
    # res_label = torch.tensor(self.labels[idx], dtype=torch.long)
    # return res.emb, res_label
    


In [220]:
# Dataset의 형태로 데이터프레임을 변환
train_ds = SBERTDataset(train_df['RawText'].tolist(), train_df['label'].tolist())
test_ds = SBERTDataset(test_df['RawText'].tolist(), test_df['label'].tolist())

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=128, shuffle=True)

In [221]:
class MLPHead(nn.Module):
    # 선형 모델에 데이터를 입력하는 형태
    # 선형 모델을 정의할때 인자값 (Linear(입력데이터의 피쳐의 수, 출력의 피쳐의 수))
    def __init__(self, input_dim, hidden=256, num_classes=2, dropout=0.2):
        super().__init__()
        # 다중 퍼셉트론층 구성
        self.net = nn.Sequential(
            # 선형 모델
            nn.Linear(input_dim, hidden),   # 입력 768 차원에서 출력은 256 차원
            nn.ReLU(),                      # 비선형 구조를 이해하기 위한 작업
            nn.Dropout(dropout),            # 과적합 방지를 위한 소실 작업
            nn.Linear(hidden, num_classes)  # 최종 출력층
            
        )
    # 순전파 함수 -> 독립변수를 받아서 예측 값을 되돌려준다.
    def forward(self, x):
        # x : 독립 변수 (document 데이터를 임베딩하고 배치로 묶은 데이터)
        result = self.net(x)    # 출력이 2차원인 확률 데이터
        return result


In [227]:
in_dim = sbert.get_sentence_embedding_dimension()
# MLPHead 모델 생성
clf = MLPHead(in_dim)
# 손실함수 -> 예측값과 실제값의 차이를 확인하는 함수
crit = nn.CrossEntropyLoss()
# 옵티마이저
opt = torch.optim.AdamW(clf.parameters(), lr= 2e-4)

In [228]:
clf.train()

for epoch in range(5):
    total = 0.0
    for x, y in train_dl:
        opt.zero_grad()
        logits = clf(x)
        loss = crit(logits, y)
        loss.backward()
        
        opt.step()
        total += loss.item() * float(x.size(0))
    print(f"epoch : {epoch}, loss : {round(total/len(train_ds), 4)}")

epoch : 0, loss : 0.6638
epoch : 1, loss : 0.6246
epoch : 2, loss : 0.5787
epoch : 3, loss : 0.5342
epoch : 4, loss : 0.4997


In [229]:
clf.eval()


y_true, y_pred = [], []


with torch.inference_mode():
    for x, y in test_dl:
        logits = clf(x)     # 예측 데이터 -> [0.xxx, 0.xxx]
        pred = logits.argmax(dim=1).tolist()     # 예측 데이터 -> [0, 1, 1, 0, ...]
        # y_true에 y를 리스트의 형태로 변환하고 데이터를 학장시킨다
        y_true.extend(y.tolist())
        y_pred += pred

print("f1_score", round(f1_score(y_true, y_pred), 4))

f1_score 0.8376


In [225]:
samples = df_na.head(10)['RawText'].tolist()


id2label = {
    0 : '부정',
    1 : '긍정'
}

@torch.no_grad()
def predict_review(
    texts, 
    batch_size = 128
):
    if isinstance(texts, str):
        texts = [texts]
    
    texts_norm = [noramlize(t) for t in texts]

    sbert.eval()
    clf.eval()

    result = []

    for idx in range(0, len(texts_norm), batch_size):
        batch_texts = texts_norm[ idx : idx + batch_size ]
        embs = sbert.encode(
            batch_texts, 
            convert_to_tensor=True, 
            normalize_embeddings=True
        )
        logits = clf(embs)
        probs = logits.softmax(dim = -1)
        preds = probs.argmax(dim=-1).tolist()

        for idx2, pred in enumerate(preds):
            prob = float( probs[idx2, pred] )
            review = texts[idx + idx2]
            label = id2label[pred]
            result.append(
                {
                    'text' : review, 
                    'prob' : prob, 
                    'label' : label
                }
            )
    return result

In [226]:
out_data = predict_review(samples)
out_data

[{'text': '귀에서 자꾸 빠져요.귀에 꼽는재 질이 미끄러운 재질이라 작은 소품이지만 재질만 바꾸어도 안 빠질것 같은데 000 것은 귀에 꼽으면 안 빠져서 사용할 때 아무 문제 없고 멍멍한 현상도 없는데 아무튼 만드는 이가 000 것과 사용 비교 해보고 하다 못해 재질이라도 바꾸든지 반품도 못하고 통화 중 몇 번씩 빠져서 사용 불가능 합니다.통화음은 상대방쪽 발음이 잘 안들린다 하고 음악 들을때는 스테레오 됩니다.통화시 사용하는 사람에게는 음질이 안 좋아 비추천내쪽에서 말하는것이 상대가 잘 발음이 안들린다 해서 중간에 통화를 중단하는 경우가 여러번 대기업 회사에서 시착도 안해보고 완성된 물건이라 판매 하는지그냥 만드는건지 ...오픈해서 반품도 못하고 너무하네요.',
  'prob': 0.8406080007553101,
  'label': '부정'},
 {'text': '아이를 출산한 기념으로 TV를 바꿨습니다. 그전에 쓰던 TV가 꽤나 무거워서 떨어지거나 하면 아이가 다칠까 봐 걱정이 되었거든요. 그런데 이 TV는 마감 퀄리티가 별로입니다. 아이가 자칫하다 TV를 만지면 손이 베일까 봐 걱정이에요. 저희 남편은 연결을 하다가 손이 살짝 까졌습니다.전에 쓰던 TV보다 무게가 가벼워서, 아이가 혹여나 다칠 일이 줄어들지 않을까 구입한 TV인데 마감이 별로라 더 걱정이네요. 마감이 깔끔하지 못한 부분은 사포로 살짝 갈아볼까 하는데 그렇게 되면 디자인도 떨어지게 될까 봐 노심초사하는 중입니다. 처음부터 마감이 괜찮았으면 이런 걱정도 없었을 텐데 여러모로 아쉽네요.',
  'prob': 0.7854402661323547,
  'label': '부정'},
 {'text': '화면에 노이즈가 생깁니다. 저희 가족 중에 아무도 TV 화면을 건드리거나 하지 않았는데 사용한 지 한 달이 되는 지금, 갑자기 화면에 노이즈가 생기네요. 큰맘 먹고 구입한 TV라 혹여나 화면에 흠집이라도 생길까 봐 조심히 청소했습니다. 배신감이 좀 드네요.이 TV의 가장 큰 장점이 화질이라고 생